# __Demo: Chain-of-Thought Prompting for Debugging a Multi-Step Problem with Hugging Face__

## __Steps to Perform:__
Step 1: Set up the Environment and Initialize `Qwen/Qwen2.5-0.5B-Instruct`

Step 2: Define a Function to Get Completion

Step 3: Set Up the Multi-Step Bug

Step 4: Compare a Direct Prompt Against a Chain-of-Thought Prompt

Step 5: Analyze the Output

### __Step 1: Set up the Environment and Initialize `Qwen/Qwen2.5-0.5B-Instruct`__
- Installs required libraries (`transformers`, `torch`, `accelerate`).
- Initializes `Qwen/Qwen2.5-0.5B-Instruct` using half-precision (`torch.float16`) on GPU without requiring API keys or license approvals.

In [1]:
!pip install -q transformers torch accelerate

import os
import torch
from transformers import pipeline

# Initialize pipeline on GPU with float16
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
print("✅ Qwen/Qwen2.5-0.5B-Instruct loaded successfully on GPU.")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ Qwen/Qwen2.5-0.5B-Instruct loaded successfully on GPU.


### __Step 2: Define a Function to Get Completion__
- Constructs a message formatted as a chat role dictionary with the user's prompt.
- Calls the Hugging Face text-generation pipeline to get a response from `Qwen/Qwen2.5-0.5B-Instruct`.
- Sets `do_sample=False` for deterministic (non-random) greedy decoding.

In [2]:
def get_completion(prompt, max_tokens=300):
    messages = [{"role": "user", "content": prompt}]
    output = generator(messages, max_new_tokens=max_tokens, do_sample=False)
    return output[0]["generated_text"][-1]["content"].strip()

print("✅ Completion helper ready.")

✅ Completion helper ready.


### __Step 3: Set Up the Multi-Step Bug__
This is a real, reproducible multi-step bug: the code below looks correct at a glance, but fails under a specific condition. Diagnosing it requires reasoning through several steps -- what the traceback says, which line it points to, why that line fails, and what the actual fix should be -- rather than jumping straight to a guessed fix.

In [3]:
buggy_code = """
def generate_explanation(alert):
    resolved_by = getattr(alert, "resolved_by", None)
    return f"Alert {alert.id} was handled by {resolved_by.upper()}"
"""

traceback_text = """
Traceback (most recent call last):
  File "explainer.py", line 24, in generate_explanation
    return f"Alert {alert.id} was handled by {resolved_by.upper()}"
AttributeError: 'NoneType' object has no attribute 'upper'
"""

print("✅ Multi-step bug and traceback loaded.")

✅ Multi-step bug and traceback loaded.


### __Step 4: Compare a Direct Prompt Against a Chain-of-Thought Prompt__
The direct prompt below asks for a fix immediately. The chain-of-thought prompt instead asks the model to reason through the problem step by step before proposing a fix -- this is the technique to prefer for a multi-step bug, since it surfaces the actual root cause rather than a guess.

In [5]:
# Direct prompt -- asks for the fix immediately, with no reasoning steps
direct_prompt = f"""
This code raises the following error. Fix it.

Code:
{buggy_code}

Error:
{traceback_text}
"""

print("=== Direct Prompt Output ===")
print(get_completion(direct_prompt, max_tokens=1500))


=== Direct Prompt Output ===
The error you're encountering is due to trying to use an `None` value as an attribute of a class (`getattr`). In Python, when you try to access an attribute of an object that doesn't exist, you get an `AttributeError`. To fix this, you should ensure that the `resolved_by` attribute exists before attempting to use it.

Here's how you can modify your function to handle this situation correctly:

```python
def generate_explanation(alert):
    if alert.resolved_by:
        resolved_by = alert.resolved_by
        return f"Alert {alert.id} was handled by {resolved_by.upper()}"
    else:
        return f"No resolution information available for alert {alert.id}"

# Example usage:
alert1 = Alert(id=1, resolved_by="John Doe")
alert2 = Alert(id=2, resolved_by="Jane Smith")

print(generate_explanation(alert1))  # Output: Alert 1 was handled by John Doe
print(generate_explanation(alert2))  # Output: No resolution information available for alert 2
```

In this corrected 

In [7]:
# Chain-of-Thought prompt -- asks the model to reason step by step before fixing
cot_prompt = f"""
Let's debug this step by step.

Code:
{buggy_code}

Error:
{traceback_text}

Think through this in steps:
1. What does the traceback tell us about which line failed and why?
2. Which variable was None at the time of the failure, and why might it be None?
3. What is the actual root cause of the bug (not just the symptom)?
4. What is the minimal, correct fix?

Answer each step explicitly, then give the corrected function.
"""

print("=== Chain-of-Thought Prompt Output ===")
print(get_completion(cot_prompt, max_tokens=1500))


=== Chain-of-Thought Prompt Output ===
1. The traceback indicates that `resolved_by` is `None`, which means there was an attempt to use an attribute on an `NoneType`. This suggests that either `alert` or `resolved_by` might not have been properly initialized before being used in the function.

2. The variable `None` is likely `NoneType` because when you try to access an attribute of `None`, Python raises an `AttributeError`.

3. The actual root cause of the bug is that `alert` and `resolved_by` were both `None` at the point where they were attempted to be accessed. This could happen if one of them was set to `None` during initialization but later changed its value.

4. To address this issue, we need to ensure that `alert` and `resolved_by` are always non-`None` before attempting to use their attributes. We can do this by adding checks to ensure that these variables are not `None` before calling the function.

Here is the corrected code with added checks for `alert` and `resolved_by`:



### __Step 5: Analyze the Output__
Compare the two responses: the direct prompt's fix may happen to be correct, but it gives no way to verify the model understood *why* the bug occurred. The chain-of-thought response should explicitly name the root cause -- that `resolved_by` is `None` when an alert has not yet been acknowledged, and `.upper()` is called on it unconditionally -- before proposing the same or a better fix. For a multi-step bug, always prefer the response you can verify reasoning from, not just the one that looks plausible.